# CNN - Regression

In [6]:
import torch
import torch.nn as nn
import torchvision.models as models

class WheelCenterRegressor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # Use ResNet18 as backbone
        self.backbone = models.resnet18(pretrained=pretrained)
        # Replace final layer with regression head
        self.backbone.fc = nn.Sequential(
            nn.Linear(self.backbone.fc.in_features, 128),
            nn.ReLU(),
            nn.Linear(128, 2),  # Output: normalized x, y
            nn.Sigmoid()        # Ensure output is in [0, 1]
        )

    def forward(self, x):
        return self.backbone(x)

# DataSet

In [7]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import json
import os

class WheelDataset(Dataset):
    def __init__(self, json_path, img_dir, transform=None):
        with open(json_path) as f:
            self.labels = json.load(f)
        self.img_dir = img_dir
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor()
        ])
        self.samples = [e for e in self.labels if e["is_present"] == 1.0]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        entry = self.samples[idx]
        img_path = os.path.join(self.img_dir, entry["filename"])
        img = Image.open(img_path).convert("RGB")
        x, y = entry["center"]
        return self.transform(img), torch.tensor([x, y], dtype=torch.float32)

# Training

In [9]:
from torch.utils.data import DataLoader
import torch.optim as optim

# Setup
model = WheelCenterRegressor()
dataset = WheelDataset("labeled_example_data/labels.json", "labeled_example_data/")
loader = DataLoader(dataset, batch_size=32, shuffle=True)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training
model.train()
for epoch in range(10):
    total_loss = 0
    for imgs, targets in loader:
        preds = model(imgs)
        loss = loss_fn(preds, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

Epoch 1 | Loss: 0.2265
Epoch 2 | Loss: 0.0706
Epoch 3 | Loss: 0.0399
Epoch 4 | Loss: 0.0409
Epoch 5 | Loss: 0.0296
Epoch 6 | Loss: 0.0288
Epoch 7 | Loss: 0.0259
Epoch 8 | Loss: 0.0277
Epoch 9 | Loss: 0.0303
Epoch 10 | Loss: 0.0202
